# Tutorial 04: Youtube and Tiktok API
Author: Maximilian Kreutner

In the tutorial this week we will learn how to use the YouTube Data API to search videos, extract statistics and manage how to handle API limits.

<!-- ## What we will cover:
1. Setting up the API Client and Securing your API Key
2. Understanding API Quotas
3. Searching for videos
4. **Basic Function 2:** Getting specific video statistics
5. **Best Practice:** Handling Pagination -->

You need to install the following modules for this tutorial.

In [ ]:
# Install the necessary dependencies
# !pip install google-api-python-client python-dotenv pandas isodate

## Youtube

### Creating and loading an API key

To access public data (like searching for videos, reading public comments, or getting channel statistics), you need an API Key.

1. Go to the [Google Cloud Console](https://console.cloud.google.com/).

2. Click on the project drop-down at the top left and click New Project. Give it a name and create it.

3. Once the project is created, make sure it is selected.

4. In the left sidebar, go to APIs & Services > Library.

5. Search for YouTube Data API v3 and click Enable.

6. Go to APIs & Services > Credentials.

7. Click Create Credentials at the top and select API Key.

Copy your new API key. Do not share it with anyone!

For many APIs you will get your own API key. 
In projects with bigger teams you have to decide if either everyone uses the same API key or all of you use their own.

If everyone uses their own key, good practice is to load the environment from a `.env` file that for example looks like this:

``
YOUTUBE_API_KEY=AIYOURAPIKEYHERE
``

Then you add the .env file to the .gitignore, so you don't upload it onto the public GitHub server.

In [ ]:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build

# This looks for a .env file and loads the variables inside it
load_dotenv()

API_KEY = os.getenv('YOUTUBE_API_KEY')
api_service_name = "youtube"
api_version = "v3"


if not API_KEY:
    print("Error: API Key not found. Make sure your .env file is set up correctly!")
else:
    youtube = build(api_service_name, api_version, developerKey=API_KEY)
    print("API Key loaded securely and YouTube Client created!")

API Key loaded securely and YouTube Client created!


### YouTube API Quotas

API Usage is usually either limited or costly.

For Youtube ach free API key receives **10,000 Quota Units per day** for free. However, different actions cost different amounts of units:
* Searching for videos (`search().list`): **100 units**
* Reading video/channel details (`videos().list`): **1 unit**
* Reading comments (`commentsThreads().list`): **1 unit**

You can find a full list of cost for each request on here: https://developers.google.com/youtube/v3/determine_quota_cost

### Searching for Videos

You can search for videos directly within the API.

Let's see how the online presence on youtube of our university looks like.

We want to search for the top 5 videos that show up when searching for 'University of mannheim'. For this we can create a API request with `youtube.search().list` with `type=video`.

Then we can get a response with `request.execute()`.

Note that searching videos is a lot more expensive than getting information about videos where you already know the URL. So if you know beforehand which videos you want to analyze you can save a lot of API Quota units.

In [ ]:
query = "University of mannheim"
max_results = 5

request = youtube.search().list(
    q=query,
    part='id,snippet',      # We only want the basic info (title, description, channel)
    maxResults=max_results,
    type='video'         # We only want videos, not playlists or channels
    # We can also define optional parameters:
    # videoDuration='short' # only get short videos
    # videoDefinition='high' # only get HD videos 
)

response = request.execute()

In [4]:
response

{'kind': 'youtube#searchListResponse',
 'etag': '1MXjYW9fgtW5hjv2YAYLK4DYmcg',
 'nextPageToken': 'CAUQAA',
 'regionCode': 'DE',
 'pageInfo': {'totalResults': 267099, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#searchResult',
   'etag': 'MEu9fd5Jwf4Va122A_BJz2lh_QI',
   'id': {'kind': 'youtube#video', 'videoId': 'BPH0HGVFhfQ'},
   'snippet': {'publishedAt': '2025-08-17T19:20:26Z',
    'channelId': 'UCBtDPBJ0qTWNqx9RHtDedCg',
    'title': 'BWL in Mannheim: 7 Dinge, die ich VOR dem Studium gerne gewusst hätte',
    'description': 'Ich bin mittlerweile im 5. Semester an der Uni Mannheim und möchte mit euch teilen, welche 7 Dinge ich gerne vor meinem ...',
    'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/BPH0HGVFhfQ/default.jpg',
      'width': 120,
      'height': 90},
     'medium': {'url': 'https://i.ytimg.com/vi/BPH0HGVFhfQ/mqdefault.jpg',
      'width': 320,
      'height': 180},
     'high': {'url': 'https://i.ytimg.com/vi/BPH0HGVFhfQ/hqdefault.jpg',
      'width':

We should now have a response that contains information about 5 videos in in JSON format.

Transform that data into a `pd.DataFrame`, where each row contains the `Title`, `Description`, `Video ID` and the `Channel` of the video.

For this we can loop through the response with `response.get('items', [])`

We can also get the thumbnails of each video and display it with `IPython.display`. We could for example analyze Thumbnails this way.

In [ ]:
from IPython.display import Image, display
import pandas as pd

videos = []

for item in response.get('items',[]):
    video_data = {
        'title': item['snippet']['title'],
        'description': item['snippet']['description'],
        'video_ID': item['id']['videoId'],
        'channel': item['snippet']['channelTitle']
    }

    thumbnail_url = item['snippet']['thumbnails']['high']['url']

    videos.append(video_data)
    display(Image(url=thumbnail_url, width=320))
    print(f"Title: {video_data['title']}")
    print(f"Description: {video_data['description']}")
    print(f"Channel: {video_data['channel']}")

    # You can get the url by concatenating with the web adress
    # You could then also download the videos or audios via pytubefix: (https://github.com/JuanBindez/pytubefix)
    print(f"URL: https://www.youtube.com/watch?v={video_data['video_ID']}")
    print("-" * 50)

df = pd.DataFrame(videos)

Title: BWL in Mannheim: 7 Dinge, die ich VOR dem Studium gerne gewusst hätte
Description: Ich bin mittlerweile im 5. Semester an der Uni Mannheim und möchte mit euch teilen, welche 7 Dinge ich gerne vor meinem ...
Channel: Finn
URL: https://www.youtube.com/watch?v=BPH0HGVFhfQ
--------------------------------------------------


Title: Wie ist das, an der Uni Mannheim zu studieren? (VLOG)
Description: Interview mit den beiden Uni Mannheim Erstsemester-Studenten: https://youtu.be/BfNN7jxtm1Q?si=IlBN_P4XGoDeoaWG Jetzt ...
Channel: David Döbele
URL: https://www.youtube.com/watch?v=zctvT9zt_iE
--------------------------------------------------


Title: So ist das Jura Studium an der Uni Mannheim
Description: Tonia und Louis geben dir einen Einblick ins Jurastudium an der Uni Mannheim. Das Besondere: Du kannst das 1. Staatsexamen ...
Channel: Universität Mannheim - University of Mannheim
URL: https://www.youtube.com/watch?v=g2pmAGuQ9cQ
--------------------------------------------------


Title: Welcome to University of Mannheim
Description: Welcome to the University of Mannheim! Discover campus life at one of Germany's leading universities. What awaits you: ...
Channel: Universität Mannheim - University of Mannheim
URL: https://www.youtube.com/watch?v=arSXLmbenzQ
--------------------------------------------------


Title: The Business School of the University of Mannheim
Description: The Business School of the University of Mannheim is one of the most renowned business schools in Europe and continuously ...
Channel: University of Mannheim Business School
URL: https://www.youtube.com/watch?v=ned46-urYJk
--------------------------------------------------


### Video Statistics
Search results only give us the `snippet` (titles, descriptions, thumbnails). What if we want to know how many **views** or **likes** a video has?

We have to use a different endpoint: `videos().list()`.

*This has a way cheaper cost: 1 quota unit per list call.*
And we can get a total of 50 videos in each call.
*Every day we can get information for 500.000 videos for free.*

Implement the method `get_video_stats` that takes previous `df` as input and appends `likes`, `duration`, `views`, `favorites`, and `comments` to each row.

In [ ]:
def get_video_stats(df, youtube):
    all_stats = []

    video_ids = df['video_ID'].tolist()
    
    for i in range(0, len(video_ids), 50):
        # Grab the next 50 IDs
        chunk = video_ids[i:i+50]
        
        # Join them together with a comma (e.g., "id1,id2,id3...")
        ids_string = ','.join(chunk)
        
        # Make ONE request for all 50 videos
        request = youtube.videos().list(
            part="statistics,contentDetails",
            id=ids_string
        )
        response = request.execute()
        
        for item in response.get('items',[]):
            stats = item.get('statistics', {})
            content = item.get('contentDetails', {})
            
            all_stats.append({
                'video_ID': item['id'],
                'views': int(stats.get('viewCount', 0)), # Youtube returns str by default, we can cast it.
                'likes': int(stats.get('likeCount', 0)),
                'favorites': int(stats.get('favoriteCount', 0)),
                'comments': int(stats.get('commentCount', 0)), # If we do not get an amount put zero
                'duration': content.get('duration', 'N/A') # Returns ISO 8601 duration string like: 'PT5M30S'
            })

    stats_df = pd.DataFrame(all_stats)
    
    # We can merge the two dataframes together by the video_ID
    merged_df = pd.merge(df, stats_df, on='video_ID', how='left')
    
    return merged_df


df_statistics = get_video_stats(df, youtube)

To parse the correct duration we can use `isodate.parse_duration`.

In [19]:
import isodate

df_statistics.duration = df_statistics.duration.apply(isodate.parse_duration)

We now have a full dataframe that contains information about views, likes and comments for our videos.

In [20]:
df_statistics

,title,description,video_ID,channel,views,likes,favorites,comments,duration
0,"BWL in Mannheim: 7 Dinge, die ich VOR dem Stud...",Ich bin mittlerweile im 5. Semester an der Uni...,BPH0HGVFhfQ,Finn,2623,102,0,38,0 days 00:08:11
1,"Wie ist das, an der Uni Mannheim zu studieren?...",Interview mit den beiden Uni Mannheim Erstseme...,zctvT9zt_iE,David Döbele,18817,392,0,27,0 days 00:14:04
2,So ist das Jura Studium an der Uni Mannheim,Tonia und Louis geben dir einen Einblick ins J...,g2pmAGuQ9cQ,Universität Mannheim - University of Mannheim,1037,37,0,2,0 days 00:03:10
3,Welcome to University of Mannheim,Welcome to the University of Mannheim! Discove...,arSXLmbenzQ,Universität Mannheim - University of Mannheim,17108,50,0,0,0 days 00:01:02
4,The Business School of the University of Mannheim,The Business School of the University of Mannh...,ned46-urYJk,University of Mannheim Business School,28451,0,0,16,0 days 00:02:07


### Video Comments

These statistics tell us how much reach a certain video has. We can also infer how popular a video is from the ratio of views/likes.

Another interesting aspect is the sentiment in the comments. Are comments favorable or unvaforable towards the video?

For this we can use the method `youtube.commentThreads().list`.

Implement a method get_video_comments, that takes as input the `df_statistics` and creates a new DataFrame that contains information about the comment (e.g. `comment_id`, `author`, `comment_text`, `like_count`, `published_at`) and for each comment records the `video_ID` the comment is from.

*We can get up to 100 top level comments in one request at a cost of 1 quota.*

If there are more than 100 comments on a video we have to go to the next page via the `nextPageToken`.

In [ ]:
def get_video_comments(df_statistics, youtube):
    all_comments = []

    for _, row in df_statistics.iterrows():
        video_id = row["video_ID"]

        # skip requests for videos with no comments
        # If you want to get information about if a video disabled comments
        # You would run youtube.commentThreads().list and get a
        # 403 ERROR with the message "commentsDisabled"
        if row["comments"] == 0:
            continue

        next_page_token = None

        while True:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                pageToken=next_page_token,
                textFormat="plainText"
            )
            response = request.execute()

            for item in response.get("items", []):
                comment_snippet = item["snippet"]["topLevelComment"]["snippet"]

                all_comments.append({
                    "video_ID": video_id,
                    "comment_id": item["snippet"]["topLevelComment"]["id"],
                    "author": comment_snippet.get("authorDisplayName"),
                    "comment_text": comment_snippet.get("textDisplay"),
                    "like_count": comment_snippet.get("likeCount"),
                    "published_at": comment_snippet.get("publishedAt"),
                    "updated_at": comment_snippet.get("updatedAt")
                })
            
            # If we cannot get all comments in one request we have to use
            # next_page_token to get to the next page of comments
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break

    return pd.DataFrame(all_comments)

df_comment = get_video_comments(df_statistics, youtube)

We can now look at the comments and see if a certain video has positive or negative comments, and what topics people are talking about.

In future exercises we will look at methods to analyze text and comments at scale.

In [25]:
df_comment.head(5)

,video_ID,comment_id,author,comment_text,like_count,published_at,updated_at
0,BPH0HGVFhfQ,UgxtfPO_aNZJAcvFJch4AaABAg,@thegreatestkid8817,Ein wirklich hochqualitatives Video! Komme hof...,1,2026-03-07T17:28:05Z,2026-03-07T17:28:05Z
1,BPH0HGVFhfQ,Ugz02W81hsoI5MRt-WR4AaABAg,@yslmanager,"Hallo Finn,\nein sehr inhaltreiches Video, dan...",0,2026-02-22T19:48:58Z,2026-02-22T19:48:58Z
2,BPH0HGVFhfQ,UgzXcArtps1QKqKguu94AaABAg,@OptimoSapiens,"Servus Finn,\nsuper Video erstmal. Ich muss no...",0,2026-02-16T07:21:32Z,2026-02-16T07:22:25Z
3,BPH0HGVFhfQ,UgzXTzeOBIzsS3fQqUR4AaABAg,@avinius5853,"Super interessantes und gut gemachtes Video, v...",0,2026-02-14T20:00:06Z,2026-02-14T20:00:06Z
4,BPH0HGVFhfQ,UgxsYX2iRlYUgZYZy4x4AaABAg,@slaiven1180,wie sieht es mit VWL in Mannheim aus? Hast du ...,0,2026-02-05T22:10:04Z,2026-02-05T22:10:04Z


In [26]:
df_comment.head(5).comment_text.to_list()

['Ein wirklich hochqualitatives Video! Komme hoffentlich im Herbstsemester auch an die Uni Mannheim und kann mir ein eigenes Bild machen.',
 'Hallo Finn,\nein sehr inhaltreiches Video, danke dafür! Ich bin zurzeit Schüler an einem Wirtschaftsgymnasium und habe großes Interesse an BWL, da es mir im Unterricht sehr leicht fällt, dafür Interesse zu finden und sich hinzusetzen und zu lernen. Das ich auf einem Wirtschaftsgymnasium bin sehe ich es als ein großes Sprungbrett. Würdest du mir raten direkt nach dem Abi mich in das Wintersemester einzutragen oder empfiehlst du doch lieber erstmal durch Praktikas Erfahrung sammeln oder WorkanTravel? Ebenfalls habe ich mich gefragt ob es notwendig ist an eine renommierte Hochschule zu gehen oder nicht? Ich selber habe sehr gute Noten, BwL:13,Mathe:13,Englisch:13…\nLetztens würde ich gerne wissen ob du denkst, dass BWL Zukunftssicher ist und was du zu dem Klischee: BWL ist überlaufen sagst.\n\nLG\n\nLG',
 'Servus Finn,\nsuper Video erstmal. Ich muss

## TikTok

Unfortunately Tiktok's official [Research API](https://developers.tiktok.com/products/research-api/) is quite restrictive.
It is unlikely that you will get access to it (however I encourage you to try and tell me if it works out for you).

There are unofficial alternatives that are based on scraping: https://github.com/davidteather/TikTok-Api
The repository contains getting started tutorials and also a deeper dive into how it works.

However, keep in mind that TikTok is aware of such unofficial APIs and might ban you if you try to do this with your own personal account.